In [174]:
#FANUC 表格擷取
import pdfplumber
import pandas as pd
import re

path = R"C:\Users\e11338\Desktop\銀泰目錄分割\FANUC ai-D伺服馬達 切割 32_130(頁數).pdf"

# 表格偵測參數 (保持 lines 策略)
table_settings = {
    "vertical_strategy": "lines",   
    "horizontal_strategy": "lines", 
    "snap_tolerance": 3,
    "join_tolerance": 2,
}

page_results = []

with pdfplumber.open(path) as pdf:
    for i, page in enumerate(pdf.pages):
        print(f"--- 正在處理第 {i+1} 頁 ---")
        
        # 1. 全頁文字掃描型號
        page_text = page.extract_text()
        
        # 使用 Regex 尋找 "Model" 或 "Model ai" 開頭的文字
        # 捕捉 Model 之後直到行末的字串
        model_match = re.search(r'Model\s+([αa-zA-Z0-9\s\.\/i_-]+)', page_text)
        
        if model_match:
            # 取得匹配到的內容，並只取第一行（避免抓到下面的描述）
            full_model_line = model_match.group(1).split('\n')[0].strip()
            # 過濾掉可能混入的標題文字 (例如 "Data sheet")
            page_model = full_model_line.replace("Data sheet", "").strip()
        else:
            page_model = "Unknown_Model"
            
        print(f"成功定位型號: {page_model}")
        
        # 2. 擷取表格
        # 使用 find_tables 確保我們抓到的是主要的規格表
        found_tables = page.find_tables(table_settings)
        
        if found_tables:
            # 取得該頁面積最大的表格（通常就是規格表）
            main_table = sorted(found_tables, key=lambda x: x.bbox[2]*x.bbox[3], reverse=True)[0]
            df = pd.DataFrame(main_table.extract())
            
            page_results.append({
                "model": page_model,
                "df": df,
                "page": i + 1
            })
            print(f"表格擷取成功，行數: {len(df)}")
        else:
            print(f"第 {i+1} 頁找不到表格")

# --- 最終結果確認 ---
print("\n" + "="*50)
if page_results:
    for res in page_results:
        print(f"頁碼: {res['page']} | 型號: {res['model']}")
    
    print("\n[數據初步預覽 - 第一頁]")
    # 這裡顯示前幾行，看看資料有沒有對齊
    print(page_results[0]['df'].iloc[:10, :4])

--- 正在處理第 1 頁 ---
成功定位型號: αiS 0.2/8000-D
表格擷取成功，行數: 16
--- 正在處理第 2 頁 ---
成功定位型號: αiS 0.3/8000-D
表格擷取成功，行數: 16
--- 正在處理第 3 頁 ---
成功定位型號: αiS 0.5/8000-D
表格擷取成功，行數: 16
--- 正在處理第 4 頁 ---
成功定位型號: αiS 1/8000-D
表格擷取成功，行數: 16
--- 正在處理第 5 頁 ---
成功定位型號: αiS 1.5/8000-D
表格擷取成功，行數: 16
--- 正在處理第 6 頁 ---
成功定位型號: αiS 2/5000-D
表格擷取成功，行數: 16
--- 正在處理第 7 頁 ---
成功定位型號: αiS 2/6000-D
表格擷取成功，行數: 16
--- 正在處理第 8 頁 ---
成功定位型號: αiS 2/8000-D
表格擷取成功，行數: 16
--- 正在處理第 9 頁 ---
成功定位型號: αiS 4/5000-D
表格擷取成功，行數: 16
--- 正在處理第 10 頁 ---
成功定位型號: αiS 4/6000-D
表格擷取成功，行數: 16
--- 正在處理第 11 頁 ---
成功定位型號: αiS 4/8000-D
表格擷取成功，行數: 16
--- 正在處理第 12 頁 ---
成功定位型號: αiS 8/3000-D
表格擷取成功，行數: 16
--- 正在處理第 13 頁 ---
成功定位型號: αiS 8/4000-D
表格擷取成功，行數: 16
--- 正在處理第 14 頁 ---
成功定位型號: αiS 8/6000-D
表格擷取成功，行數: 16
--- 正在處理第 15 頁 ---
成功定位型號: αiS 8/8000-D
表格擷取成功，行數: 16
--- 正在處理第 16 頁 ---
成功定位型號: αiS 12/2000-D
表格擷取成功，行數: 16
--- 正在處理第 17 頁 ---
成功定位型號: αiS 12/3000-D
表格擷取成功，行數: 16
--- 正在處理第 18 頁 ---
成功定位型號: αiS 12/4000-D
表格擷取成功，行數: 16
--- 正在處理第 19 頁 ---
成功定位型號:

In [30]:
page_results[0]['df']

,0,1,2,3,4,5,6,7,8,9,10,11
0,,Item,,,Symbol,,,Value,,,Unit,
1,Continuous torque (at low speed) (*),NaN,NaN,Tc,NaN,NaN,0.16\n1.6,NaN,NaN,Nm\nkgfcm,NaN,NaN
2,Continuous current (at low speed) (*),NaN,NaN,Ic,NaN,NaN,0.79,NaN,NaN,A (rms),NaN,NaN
3,Rated output (*),NaN,NaN,Pr,NaN,NaN,0.050\n0.067,NaN,NaN,kW\nHP,NaN,NaN
4,Rated rotation speed,NaN,NaN,Nr,NaN,NaN,8000,NaN,NaN,min-1,NaN,NaN
5,Maximum rotation speed,NaN,NaN,Nmax,NaN,NaN,8000,NaN,NaN,min-1,NaN,NaN
6,Maximum torque (*),NaN,NaN,Tmax,NaN,NaN,0.55\n5.6,NaN,NaN,Nm\nkgfcm,NaN,NaN
7,Moment of inertia of rotor,NaN,NaN,Jm,NaN,NaN,0.00000340\n0.0000347,NaN,NaN,kgm2\nkgfcms2,NaN,NaN
8,Moment of inertia of rotor (with brake),NaN,NaN,Jm,NaN,NaN,0.00000450\n0.0000459,NaN,NaN,kgm2\nkgfcms2,NaN,NaN
9,Torque constant (*),NaN,NaN,Kt,NaN,NaN,0.20\n2.0,NaN,NaN,Nm/A (rms)\nkgfcm/A (rms),NaN,NaN


In [ ]:
#表格清洗
import pandas as pd
import numpy as np

# 假設 res['df'] 是你截圖中的那個 12 欄 DataFrame
def clean_fanuc_table(df_raw, model_name):
    # 1. 識別關鍵欄位索引
    # 根據截圖：Item=0, Symbol=3, Value=6, Unit=9
    # 我們使用 iloc 提取這四個主要資訊欄位
    df = df_raw.iloc[:, [0, 3, 6, 9]].copy()
    df.columns = ['Item', 'Symbol', 'Value', 'Unit']
    
    # 2. 基本清理：移除全空行，並將內容轉為字串處理
    df = df.replace(r'^\s*$', np.nan, regex=True)
    
    # 移除首行如果是標題的情況
    if "Item" in str(df.iloc[0, 0]):
        df = df.iloc[1:].reset_index(drop=True)
        
    # 3. 處理換行符拆分 (Explode)
    # 這是最關鍵的一步：將 "0.16\n1.6" 拆成列表，再展開成兩行
    df['Value'] = df['Value'].astype(str).str.split(r'\n')
    df['Unit'] = df['Unit'].astype(str).str.split(r'\n')
    
    # 使用 pandas 的 explode 功能將列表展開
    # 注意：Value 和 Unit 必須長度一致才能同時 explode (在 FANUC 型錄中通常是一致的)
    df = df.explode(['Value', 'Unit'])
    
    # 4. 清理與填充
    # 去除前後空白，並處理 explode 後產生的空值
    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
    df['Item'] = df['Item'].replace(['None', 'nan', ''], np.nan).ffill()
    df['Symbol'] = df['Symbol'].replace(['None', 'nan', ''], np.nan).ffill()
    
    # 過濾掉 Value 為空的無效行
    df = df[df['Value'].notna() & (df['Value'] != 'nan') & (df['Value'] != '')]
    
    # 5. 加上 Model 標籤
    df.insert(0, 'Model', model_name)
    
    return df

# 執行清理
all_cleaned_dfs = []
for res in page_results:
    cleaned_df = clean_fanuc_table(res['df'], res['model'])
    all_cleaned_dfs.append(cleaned_df)

# 合併最終結果
final_df = pd.concat(all_cleaned_dfs, ignore_index=True)
def format_precision(val):
    try:
        # 先轉為浮點數，確保它是數值格式
        f_val = float(val)
        # 使用格式化字串轉為字串（避免科學記號）
        # 去除右側的 '0'，如果最後剩下小數點也一併去除
        s_val = "{:.10f}".format(f_val).rstrip('0').rstrip('.')
        return s_val
    except (ValueError, TypeError):
        return val
    
final_df["Value"] = final_df["Value"].apply(format_precision)

print("--- 馬達型錄清洗結果 ---")
display(final_df.head(22))

--- 馬達型錄清洗結果 ---


,Model,Item,Symbol,Value,Unit
0,αiS 0.2/8000-D,Continuous torque (at low speed) (*),Tc,0.16,Nm
1,αiS 0.2/8000-D,Continuous torque (at low speed) (*),Tc,1.6,kgfcm
2,αiS 0.2/8000-D,Continuous current (at low speed) (*),Ic,0.79,A (rms)
3,αiS 0.2/8000-D,Rated output (*),Pr,0.05,kW
4,αiS 0.2/8000-D,Rated output (*),Pr,0.067,HP
5,αiS 0.2/8000-D,Rated rotation speed,Nr,8000,min-1
6,αiS 0.2/8000-D,Maximum rotation speed,Nmax,8000,min-1
7,αiS 0.2/8000-D,Maximum torque (*),Tmax,0.55,Nm
8,αiS 0.2/8000-D,Maximum torque (*),Tmax,5.6,kgfcm
9,αiS 0.2/8000-D,Moment of inertia of rotor,Jm,0.0000034,kgm2


In [ ]:
uni = final_df["Model"].unique().tolist()
uni

In [171]:
uni_item = final_df["Item"].unique().tolist()
uni_item

['Continuous torque (at low speed) (*)',
 'Continuous current (at low speed) (*)',
 'Rated output (*)',
 'Rated rotation speed',
 'Maximum rotation speed',
 'Maximum torque (*)',
 'Moment of inertia of rotor',
 'Moment of inertia of rotor (with brake)',
 'Torque constant (*)',
 'Winding resistance (between terminals) (*)',
 'Thermal time constant',
 'Static friction',
 'Weight',
 'Weight (with brake)',
 'Max. current of servo amp.',
 'Moment of inertia of rotor (with 35Nm brake)',
 'Weight (with 35Nm brake)',
 'Moment of inertia of rotor (with 70Nm brake)',
 'Weight (with 70Nm brake)',
 'Rated Speed']

In [142]:
import pandas as pd

# 1. 目標參數字典 (維持不變，作為篩選基準)
target_dict = {
    "Item": [
        "Continuous torque (at low speed) (*)", 
        "Maximum rotation speed", 
        "Moment of inertia of rotor", 
        "Moment of inertia of rotor (with brake)",
        'Moment of inertia of rotor (with 35Nm brake)',
        'Moment of inertia of rotor (with 70Nm brake)',


    ],
    "Unit": ["Nm", "min-1", "kgm2", "kgm2", "kgm2", "kgm2"]
}

filter_df = pd.DataFrame(target_dict)

# 2. 進行篩選
filtered_df = pd.merge(final_df, filter_df, on=['Item', 'Unit'], how='inner')

# 3. 執行矩陣翻轉 (直接使用 Item 作為 Columns)
# 這樣標題就會直接是 "Continuous torque (at low speed)" 等字串
summary_df = filtered_df.pivot_table(
    index='Model', 
    columns='Item', 
    values='Value',
    aggfunc='first'
).reset_index()

# 4. 清理與型態轉換
summary_df.columns.name = None
cols = [c for c in summary_df.columns if c != 'Model']
summary_df[cols] = summary_df[cols].apply(pd.to_numeric, errors='coerce')

# 5. (選擇性) 如果你想要把標題改得更短或更符合計算需求
# 你可以使用 rename 進行最後的自訂映射
rename_map = {
    "Continuous torque (at low speed) (*)": "Torque_Nm",
    "Maximum rotation speed": "Max_RPM",
    "Moment of inertia of rotor": "Inertia_kgm2",
    "Moment of inertia of rotor (with brake)": "Inertia_Brake_kgm2",
    'Moment of inertia of rotor (with 35Nm brake)': "Inertia_Brake_35Nm_kgm2",
    'Moment of inertia of rotor (with 70Nm brake)': "Inertia_Brake_70Nm_kgm2"
}
summary_df = summary_df.rename(columns=rename_map)

#====================================順序重排==================================

# 1. 從 final_df 提取型號的原始出現順序 (維持讀取時的先後)
# unique() 會按照資料出現的先後順序回傳
original_order = final_df['Model'].unique().tolist()

# 2. 將 summary_df 的 Model 欄位轉為 Categorical 型態，並指定原始順序
summary_df['Model'] = pd.Categorical(
    summary_df['Model'], 
    categories=original_order, 
    ordered=True
)

# 3. 根據這個 Categorical 順序進行排序
summary_df = summary_df.sort_values('Model').reset_index(drop=True)

# 4. (選填) 確保欄位標題也按照你字典定義的順序排列，方便計算
desired_columns = ['Model', 'Torque_Nm', 'Max_RPM', 'Inertia_kgm2', 'Inertia_Brake_kgm2', "Inertia_Brake_35Nm_kgm2", "Inertia_Brake_70Nm_kgm2"]
# 只取實際存在的欄位避免報錯
summary_df = summary_df[[col for col in desired_columns if col in summary_df.columns]]

print("--- 最終順序校正後的查找表 ---")
display(summary_df)

--- 最終順序校正後的查找表 ---


,Model,Torque_Nm,Max_RPM,Inertia_kgm2,Inertia_Brake_kgm2,Inertia_Brake_35Nm_kgm2,Inertia_Brake_70Nm_kgm2
0,αiS 0.2/8000-D,0.16,8000,0.000003,0.000005,NaN,NaN
1,αiS 0.3/8000-D,0.32,8000,0.000007,0.000008,NaN,NaN
2,αiS 0.5/8000-D,0.65,8000,0.000026,0.000035,NaN,NaN
3,αiS 1/8000-D,1.20,8000,0.000048,0.000057,NaN,NaN
4,αiS 1.5/8000-D,1.60,8000,0.000071,0.000081,NaN,NaN
...,...,...,...,...,...,...,...
88,αiF 22/4000HV-D,22.00,4000,0.011500,NaN,0.0121,NaN
89,αiF 30/3000HV-D,30.00,3000,0.017100,NaN,0.0177,NaN
90,αiF 30/4000HV-D,30.00,4000,0.017100,NaN,0.0177,NaN
91,αiF 40/3000HV-D,38.00,3000,0.022700,NaN,0.0233,0.0236


In [146]:
with pd.ExcelWriter(R'C:\Users\e11338\Desktop\Feed System GAI\data\FANUC_Specs.xlsx', engine='openpyxl') as writer:
    final_df.to_excel(writer, sheet_name='ALL', index=False)
    summary_df.to_excel(writer, sheet_name='Model', index=False)

In [149]:
### 文本擷取

import fitz  # PyMuPDF
import re


def step1_extract_text(pdf_path):
    """
    從 PDF 中提取文字，並進行初步的格式清理。
    """
    try:
        # 開啟 PDF 檔案
        doc = fitz.open(pdf_path)
        print(f"--- 檔案讀取成功：{pdf_path} ---")
        print(f"總頁數: {len(doc)}")
        
        extracted_data = []

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # 提取文字
            raw_text = page.get_text("text")
            
            # 初步清理：移除多餘的連續空白、統一換行符
            clean_text = re.sub(r'\n\s*\n', '\n\n', raw_text) # 保持段落感
            clean_text = clean_text.strip()
            
            # 儲存結果（包含頁碼資訊，這對後續 RAG 引用非常重要）
            extracted_data.append({
                "page": page_num + 1,
                "content": clean_text
            })
            
            # 預覽前兩頁
            if page_num < 2:
                print(f"\n[第 {page_num + 1} 頁預覽]:")
                print(clean_text[:300] + "...") 
                print("-" * 30)

        doc.close()
        return extracted_data

    except Exception as e:
        print(f"讀取失敗：{e}")
        return None

# --- 執行處 ---
# 請將 'HIWIN_Catalog.pdf' 換成你實際的檔案路徑

raw_pages = step1_extract_text(R"C:\Users\e11338\Downloads\B65542EN_01_ai-D伺服馬達仕樣.pdf")

--- 檔案讀取成功：C:\Users\e11338\Downloads\B65542EN_01_ai-D伺服馬達仕樣.pdf ---
總頁數: 282

[第 1 頁預覽]:
© FANUC CORPORATION, 2023 
< SERVO MOTOR @+-D !
DESCRIPTIONS
B-65542EN/01...
------------------------------

[第 2 頁預覽]:
Important notices 
B-65542EN/01 
ii 
Important notices 

 No part of this manual may be reproduced in any form. 
 The appearance and specifications of this product are subject to change without notice. 

The products in this manual are controlled based on Japan's “Foreign Exchange and Foreign Trad...
------------------------------


In [150]:
###文本擷取確認存檔

def save_extraction_to_file(extracted_data, output_filename="FANUC_extraction_check.md"):
    """
    將擷取到的文字資料存成 Markdown 檔案，並自動計算字數。
    """
    

    try:
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write("# PMI 型錄文字擷取結果檢視\n\n")
            f.write(f"**總計擷取頁數:** {len(extracted_data)}\n\n")
            f.write("---\n\n")

            for item in extracted_data:
                # 使用 .get() 確保如果找不到鍵也不會當機，並現場計算字數
                page_no = item.get('page', '未知')
                content = item.get('content', '')
                char_count = len(content)
                
                f.write(f"## 第 {page_no} 頁\n")
                f.write(f"**本頁字數:** {char_count} 字\n\n")
                f.write("### 內容摘要:\n")
                f.write("```text\n")
                f.write(content if content else "[此頁無文字內容]")
                f.write("\n```\n")
                f.write("\n---\n\n")
        
        print(f"檢查檔案已成功生成：{output_filename}")

    except Exception as e:
        print(f"儲存失敗，原因：{e}")

# --- 執行處 ---
# 請確保這裡傳入的是你上一個步驟得到的列表變數
#save_extraction_to_file(raw_pages)


In [181]:
### 文本清理確認 --> 頁清洗、段落清洗
import re

def is_useful_page(content):
    # 1. 定義馬達規格表必現的關鍵字清單 (你的黃金清單)
    spec_gold_list = ['Continuous torque (at low speed) (*)',
        'Continuous current (at low speed) (*)',
        'Rated output (*)',
        'Rated rotation speed',
        'Maximum rotation speed',
        'Maximum torque (*)',
        'Moment of inertia of rotor',
        'Moment of inertia of rotor (with brake)',
        'Torque constant (*)',
        'Winding resistance (between terminals) (*)',
        'Thermal time constant',
        'Static friction',
        'Weight',
        'Weight (with brake)',
        'Max. current of servo amp.',
        'Moment of inertia of rotor (with 35Nm brake)',
        'Weight (with 35Nm brake)',
        'Moment of inertia of rotor (with 70Nm brake)',
        'Weight (with 70Nm brake)',
        'Rated Speed'
        ]
    
    # 2. 計算頁面中包含了多少個關鍵字
    # 使用 lower() 確保大小寫不敏感，並移除 (*) 避免正則匹配問題
    content_lower = content.lower()
    match_count = 0
    for key in spec_gold_list:
        if key.lower() in content_lower:
            match_count += 1
            
    # 3. 設定過濾門檻
    # 如果頁面出現了 80% 以上的規格欄位 (20個中的16個)，判定為馬達數據頁
    match_ratio = match_count / len(spec_gold_list)
    if match_ratio >= 0.6:
        return False # 這是數據頁，過濾掉
        
    # 4. 輔助檢查：外型尺寸圖頁面 (Outline Drawings)
    # 如果頁面包含 "OUTLINE DRAWING" 且數字極多，也過濾
    if "OUTLINE DRAWING" in content.upper() or "EXTERNAL DIMENSIONS" in content.upper():
        digit_count = len(re.findall(r'\d', content))
        if digit_count > 200: # 尺寸圖通常有無數個座標數字
            return False

    # 5. 保留原本的基礎判斷 (防止漏網之魚)
    lines = [l.strip() for l in content.split('\n') if l.strip()]
    if not lines: return False
    
#==========================================

    # 移除換行，保留空格以利單字判定
    text = content.strip()
    if len(text) < 100: return False
    
    # 1. 提取所有「單字」(長度大於等於 2 的連續英文字母)
    # 這是判斷「文」的關鍵：工程圖中的標註如 "M5", "130" 會被排除或稀釋
    words = re.findall(r'\b[a-zA-Z]{2,}\b', text)
    alpha_word_char_count = sum(len(w) for w in words)
    
    # 2. 提取所有「數據與符號」(數字、孤立字元、特殊符號)
    # 這是判斷「圖/數據」的關鍵
    data_chars = re.findall(r'[\d●○■□▪▫\-\.·]', text)
    data_char_count = len(data_chars)
    
    # 3. 計算圖文比例
    # 我們定義「文」是真正的單字組成；「圖」是數據與符號
    if (alpha_word_char_count + data_char_count) == 0: return False
    
    # 語義文字佔比
    semantic_ratio = alpha_word_char_count / (alpha_word_char_count + data_char_count)
    
    # --- 判定門檻 ---
    
    # 針對圖 8 這類工程圖頁面：
    # 它的 alpha_word_char_count 會非常低（因為大多是孤立的標註）
    # 它的 data_char_count 會非常高（因為全是尺寸數字和符號）
    
    # 如果語義文字佔比低於 30%，代表這頁「圖 (數據標註)」遠多於「文 (說明敘述)」
    if semantic_ratio < 0.3:
        return False
        
    # 輔助判定：如果數字與符號的總量是單字字母量的 2 倍以上，判定為工程圖頁
    if data_char_count > (alpha_word_char_count * 2):
        return False

    return True


def is_text_description(text):
    t = text.strip()
    if len(t) < 30: return False 

    # --- 1. 符號與無意義字元偵測 (針對圖 7) ---
    # 統計圓圈、方塊、橫線、點點、斜線
    # 這些通常是表格的格線或對應點
    junk_symbols = re.findall(r'[●○■□▪▫\-\.·]', t)
    symbol_ratio = len(junk_symbols) / len(t)
    
    # 如果特殊符號佔比超過 15%，極高機率是規格對照表
    if symbol_ratio > 0.15:
        return False

    # --- 2. 語義動詞判定 (保留說明文字的靈魂) ---
    # 即使型號很多，只要有這些動詞，就代表這是在「解釋」而非「列清單」
    meaningful_verbs = r'\b(is|are|was|were|will|should|can|has|have|provide|use|note|caution|refer|adjust|mount|connect|install|replace|set|check)\b'
    has_verbs = bool(re.search(meaningful_verbs, t.lower()))
    
    # --- 3. 型號密度與結構檢查 ---
    model_keywords = len(re.findall(r'(α|a)i[SF]', t))
    lines = [l for l in t.split('\n') if l.strip()]
    
    # 情況 A：型號出現次數與行數比例過高 (例如每一行都是型號)
    if len(lines) > 0:
        model_line_ratio = model_keywords / len(lines)
        # 如果超過 70% 的行都在講型號，且完全沒有動詞，砍掉
        if model_line_ratio > 0.7 and not has_verbs:
            return False

    # 情況 B：純數字與符號的組合 (如圖 7 中間那段)
    no_space_text = t.replace(" ", "").replace("\n", "")
    data_chars = len(re.findall(r'[\d\.\-\/●○]', no_space_text))
    if len(no_space_text) > 0:
        data_ratio = data_chars / len(no_space_text)
        # 數字加上符號佔比超過 75%，且無動詞，砍掉
        if data_ratio > 0.75 and not has_verbs:
            return False

    # --- 4. 結尾判定 ---
    # 真正的英文說明通常會以句號結尾，表格則不會
    if not t.endswith('.') and data_ratio > 0.5 and not has_verbs:
        return False

    return True

def step2_dual_layer_clean(extracted_data):
    """
    整合流程：先進行全域雜質清除，再過濾頁面與段落。
    """
    # 定義無意義的全域標籤黑名單
    global_junk_patterns = [
        r'B-65542EN/\d+',  # 匹配 B-65542EN/01, B-65542EN/02 等
        r'FANUC CORPORATION, \d{4}', # 匹配版權宣告
        r'Important notices',
        r'All rights reserved'
    ]
    
    refined_docs = []
    for item in extracted_data:
        content = item['content']
        
        # --- 0. 全域清除無意義標籤 ---
        for pattern in global_junk_patterns:
            content = re.sub(pattern, '', content, flags=re.IGNORECASE)
            
        # --- 1. 第一層：頁面過濾 (使用你提到的 80% 關鍵字或圖文比例) ---
        if not is_useful_page(content):
            continue 
            
        # --- 2. 第二層：段落過濾 (排除段落中的表格碎片) ---
        paragraphs = re.split(r'\n\s*\n', content) 
        filtered_paragraphs = [p.strip() for p in paragraphs if is_text_description(p)]
        
        clean_text = "\n\n".join(filtered_paragraphs)
        if clean_text:
            refined_docs.append({
                "page": item['page'], 
                "content": clean_text
            })
            
    return refined_docs

purified_data = step2_dual_layer_clean(raw_pages)
save_extraction_to_file(purified_data)

檢查檔案已成功生成：FANUC_extraction_check.md


In [187]:
###切片工程，chuncks可視化
import re

def semantic_chunking_optimized(purified_data, chunk_size=800, overlap=120):
    chunks = []
    min_chunk_size = 120 
    
    def fix_special_chars(text):
            """
            修正 Markdown 特殊符號並清理 PDF 雜字。
            """
            text = text.replace("~", "\~")
            text = text.replace("~~", "~")
            text = text.replace("■", "●").replace("□", "○")
            return text

    def is_garbage_chunk(content):
        """
        針對 FANUC 特徵強化的垃圾攔截邏輯
        """
        lines = [l.strip() for l in content.split('\n') if l.strip()]
        if not lines: return True

        # --- 1. 目錄與索引排除 (圖 460) ---
        if content.count("....") > 3 or "CONTENTS" in content.upper():
            return True

        # --- 2. 圓圈與特殊符號密度檢查 (針對圖 7) ---
        # 統計圓圈、方塊、分隔線符號
        symbol_pattern = r'[●○■□▪▫\-·]'
        symbols = re.findall(symbol_pattern, content)
        if len(symbols) > 5 and (len(symbols) / len(content) > 0.05):
            return True

        # --- 3. 語義動詞判定 (核心：區分句子與表格) ---
        # 增加更多安裝與維護相關的動詞
        meaningful_verbs = r'\b(is|are|was|were|will|should|can|has|have|provide|use|note|caution|refer|adjust|mount|connect|install|replace|set|check|avoid|keep)\b'
        has_verbs = bool(re.search(meaningful_verbs, content.lower()))

        # --- 4. 型號堆疊檢查 (針對圖 155) ---
        model_start_count = sum(1 for l in lines if re.match(r'^(α|a)i[SF]', l))
        model_ratio = model_start_count / len(lines) if len(lines) > 0 else 0

        # --- 5. 數據與符號綜合密度 ---
        no_space_text = content.replace(" ", "").replace("\n", "")
        data_chars = len(re.findall(r'[\d\.\-\/●○]', no_space_text))
        data_ratio = data_chars / len(no_space_text) if len(no_space_text) > 0 else 0

        # --- 判定邏輯組合 ---

        # A. 如果型號比例極高且沒有動詞，判定為型號表 (過濾圖 7)
        if model_ratio > 0.6 and not has_verbs:
            return True

        # B. 如果數據與符號佔比超過 65% 且沒有正常的英文句式，判定為碎片
        if data_ratio > 0.65 and not has_verbs:
            return True

        # C. 原有的對齊殘影檢查 (保留並強化)
        alignment_gaps = len(re.findall(r'\s{5,}', content))
        if alignment_gaps >= 3 and not has_verbs:
            return True

        return False

    for item in purified_data:
        text = fix_special_chars(item['content'])
        page_num = item['page']
        start = 0
        
        while start < len(text):
            remaining_len = len(text) - start
            
            # --- 碎片處理 ---
            if remaining_len < (chunk_size + min_chunk_size) / 2:
                chunk_content = text[start:].strip()
                
                if is_garbage_chunk(chunk_content):
                    break

                # 嘗試與前一個 chunk 合併 (如果是同一頁)
                if chunks and chunks[-1]["metadata"]["page"] == page_num:
                    chunks[-1]["content"] += "\n\n" + chunk_content
                elif len(chunk_content) > 30: 
                    chunks.append({
                        "content": chunk_content,
                        "page":page_num, 
                        "metadata": {"page": page_num, "type": "technical_manual"}
                    })
                break 

            # --- 正常切片邏輯 (以句號為語義切分點) ---
            end = start + chunk_size
            if end < len(text):
                # 尋找最近的英文句號作為切點，避免切斷句子
                last_punctuation = text.rfind('. ', start, end + 100)
                if last_punctuation != -1 and last_punctuation > start + (chunk_size // 2):
                    end = last_punctuation + 1
            
            chunk_content = text[start:end].strip()
            
            # 在正式存入前攔截垃圾，確保 chunk 具有足夠長度與意義
            if len(chunk_content) > 40 and not is_garbage_chunk(chunk_content):
                chunks.append({
                    "content": chunk_content,
                    "page":page_num, 
                    "metadata": {"page": page_num, "type": "technical_manual"}
                })
            
            start = end - overlap
            
    print(f"切片完成！共產生 {len(chunks)} 個英文語義區塊。")
    return chunks

def save_chunks_to_markdown(chunks, filename="FANUC_chunking_visualization.md"):
    def clean_line_breaks(text):
        """
        修正 PDF 換行，將被切斷的英文句子接回。
        """
        # 如果一行結尾不是句號、冒號、感嘆號，則將其與下一行合併
        fixed_text = re.sub(r'([^.:?!])\n', r'\1 ', text)
        # 處理多餘空格
        fixed_text = re.sub(r' +', ' ', fixed_text)
        return fixed_text.strip()
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write("# FANUC 語義切片可視化報告\n")
        f.write(f"總切片數: {len(chunks)} 個\n\n")
        f.write("---\n\n")
        
        for i, chunk in enumerate(chunks):
            cleaned_content = clean_line_breaks(chunk['content'])
            
            f.write(f"## Chunk-#{i+1}\n")
            source = chunk['metadata'].get('page', 'Unknown')
            f.write(f"- Source Page: Page {source}\n")
            f.write(f"- Character Count: {len(cleaned_content)}\n")
            f.write(f"- Data Type: {chunk['metadata'].get('type', 'N/A')}\n\n")
            
            f.write("> ### Content Block:\n")
            f.write("> " + cleaned_content.replace("\n", "\n> ") + "\n\n")
            f.write("---\n\n")
            
    print(f"可視化檔案已儲存至: {filename}")

# --- 執行 ---
final_chunks = semantic_chunking_optimized(purified_data)
# --- 執行儲存 ---
save_chunks_to_markdown(final_chunks)

切片完成！共產生 290 個英文語義區塊。
可視化檔案已儲存至: FANUC_chunking_visualization.md


<>:12: SyntaxWarning: invalid escape sequence '\~'
<>:12: SyntaxWarning: invalid escape sequence '\~'
C:\Users\e11338\AppData\Local\Temp\ipykernel_30340\3682814444.py:12: SyntaxWarning: invalid escape sequence '\~'
  text = text.replace("~", "\~")


In [188]:
save_extraction_to_file(final_chunks, output_filename="FANUC_final_chunks_check.md")

檢查檔案已成功生成：FANUC_final_chunks_check.md


In [189]:
import json

# 1. 將 FANUC 切片結果儲存為 JSON
# ensure_ascii=False 是為了確保 α (alpha) 等特殊符號能正常儲存而不被編碼
output_filename = "FANUC_final_chunks.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_chunks, f, ensure_ascii=False, indent=4)

print(f"FANUC 切片資料已成功儲存為 {output_filename}")

FANUC 切片資料已成功儲存為 FANUC_final_chunks.json
